## 01 — Exploration des datasets
> Exercicedb --- Daily Food Nutrition --- Diet Recommendations --- gym members exercice tracking synthetic --- gym member exerecice tracking

In [13]:
# cellule import 
import json
import os 
import pandas as pd 
from pathlib import Path
import io

### Exercicedb

In [14]:
# --- ExerciceDB ---

# Configuration des chemins
BASE_DIR = Path.cwd().parent 
DATA_RAW = BASE_DIR / "data" / "raw"
FILE_PATH = DATA_RAW / "exercisedb_hobby.json"

print(f"ANALYSE : ExerciseDB (API JSON)")

if FILE_PATH.exists():
    with open(FILE_PATH, 'r', encoding='utf-8') as f:
        raw_data = json.load(f)
    
    if isinstance(raw_data, dict) and 'data' in raw_data:
        df = pd.DataFrame(raw_data['data'])
        
        # --- RAPPORT DE STRUCTURE ---
        print(f"Succès ! {df.shape[0]} exercices chargés.")
        print(f"Dimensions : {df.shape[0]} lignes x {df.shape[1]} colonnes")
        
        # Les types de données et valeurs nulles 
        print("\n Types de données colonnes et null :")
        df.info()
        
        # Aperçu des données
        print("\n Échantillon des 5 premières lignes :")
        display(df.head(5))
        
        # Attention les listes OBJECT sont des listes qui devront être converties en string pour le sql ('targetMuscles', 'equipments', 'bodyParts')  
    else:
        print("Clé 'data' introuvable dans le JSON.")
else:
    print(f"Fichier introuvable : {FILE_PATH}")

ANALYSE : ExerciseDB (API JSON)
Succès ! 25 exercices chargés.
Dimensions : 25 lignes x 9 colonnes

 Types de données colonnes et null :
<class 'pandas.DataFrame'>
RangeIndex: 25 entries, 0 to 24
Data columns (total 9 columns):
 #   Column            Non-Null Count  Dtype 
---  ------            --------------  ----- 
 0   exerciseId        25 non-null     str   
 1   name              25 non-null     str   
 2   imageUrl          25 non-null     str   
 3   bodyParts         25 non-null     object
 4   equipments        25 non-null     object
 5   exerciseType      25 non-null     str   
 6   targetMuscles     25 non-null     object
 7   secondaryMuscles  25 non-null     object
 8   keywords          25 non-null     object
dtypes: object(5), str(4)
memory usage: 1.9+ KB

 Échantillon des 5 premières lignes :


,exerciseId,name,imageUrl,bodyParts,equipments,exerciseType,targetMuscles,secondaryMuscles,keywords
0,exr_41n2ha5iPFpN3hEJ,Bridge - Mountain Climber,https://cdn.exercisedb.dev/media/w/images/EsVO...,[WAIST],[BODY WEIGHT],STRENGTH,[OBLIQUES],"[QUADRICEPS, PECTINEUS, ADDUCTOR BREVIS, TENSO...","[Bodyweight exercise for waist, Bridge Mountai..."
1,exr_41n2haAabPyN5t8y,Side Lunge,https://cdn.exercisedb.dev/media/w/images/RB7N...,"[QUADRICEPS, THIGHS]",[BODY WEIGHT],STRENGTH,"[GLUTEUS MAXIMUS, QUADRICEPS]","[ADDUCTOR MAGNUS, GLUTEUS MEDIUS, SOLEUS, TENS...","[Body weight exercise for thighs, Quadriceps s..."
2,exr_41n2hadPLLFRGvFk,Sliding Floor Pulldown on Towel,https://cdn.exercisedb.dev/media/w/images/9fW9...,[BACK],[BODY WEIGHT],STRENGTH,[LATISSIMUS DORSI],"[POSTERIOR DELTOID, TRICEPS BRACHII, TERES MIN...","[Body weight back exercise, Sliding Floor Pull..."
3,exr_41n2hadQgEEX8wDN,Triceps Dip,https://cdn.exercisedb.dev/media/w/images/MruS...,"[TRICEPS, UPPER ARMS]",[BODY WEIGHT],STRENGTH,[TRICEPS BRACHII],"[PECTORALIS MAJOR STERNAL HEAD, PECTORALIS MAJ...","[Triceps Dip Workout, Bodyweight Tricep Exerci..."
4,exr_41n2haNJ3NA8yCE2,Dumbbell Incline One Arm Hammer Press,https://cdn.exercisedb.dev/media/w/images/qkXo...,[UPPER ARMS],[DUMBBELL],STRENGTH,[RECTUS ABDOMINIS],"[TERES MAJOR, PECTORALIS MAJOR CLAVICULAR HEAD...","[One Arm Dumbbell Press, Hammer Press Workout,..."


### daily_food_nutrition_dataset.csv
> Aliments, macros nutritionnels, type de repas

In [15]:
# --- Daily Food & Nutrition ---

# Attention certaines lignes du CSV sont mal formatées il y a des virgules en plus dans les champs 
# pandas croit que c'est une nouvelle colonne 
# Raison pour laquelle la cellule est plus complexe on calcule d'abord le nombre colonne attendues 
# Ensuite on vérifie ligne par ligne et on transforme les lignes mal formatées pour que pandas puisse lire 

food_file = DATA_RAW / "daily_food_nutrition_dataset.csv"

if food_file.exists():
    cleaned_lines = []
    header = None
    
    with open(food_file, 'r', encoding='utf-8') as f:
        header = f.readline().strip() 
        expected_count = len(header.split(','))
        
        for i, line in enumerate(f):
            parts = line.strip().split(',')
            
            # On vérif si le nombre de colonnes est sup à l'attendu 
            if len(parts) > expected_count:
                # On suppose que le problème est dans le premier champ (Food_Item)
                # On fusionne les n premiers éléments en trop pour n'en faire qu'un
                overflow = len(parts) - expected_count
                new_food_name = " ".join(parts[:overflow + 1]).replace('"', '')
                new_line = [new_food_name] + parts[overflow + 1:]
                cleaned_lines.append(",".join(new_line))
            else:
                cleaned_lines.append(line.strip())

    # On transforme les lignes nettoyées en un seul bloc de texte pour Pandas
    final_data = header + "\n" + "\n".join(cleaned_lines)
    df_food = pd.read_csv(io.StringIO(final_data))

    print(f"Chargement réussi ! {df_food.shape[0]}")
    
    # Les types de données et valeurs nulles
    df_food.info()
    display(df_food.head(25)) 

Chargement réussi ! 651
<class 'pandas.DataFrame'>
RangeIndex: 651 entries, 0 to 650
Data columns (total 12 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   Food_Item          651 non-null    str    
 1   Category           651 non-null    str    
 2   Calories (kcal)    651 non-null    int64  
 3   Protein (g)        651 non-null    float64
 4   Carbohydrates (g)  651 non-null    float64
 5   Fat (g)            651 non-null    float64
 6   Fiber (g)          651 non-null    float64
 7   Sugars (g)         651 non-null    float64
 8   Sodium (mg)        651 non-null    int64  
 9   Cholesterol (mg)   651 non-null    int64  
 10  Meal_Type          651 non-null    str    
 11  Water_Intake (ml)  651 non-null    int64  
dtypes: float64(5), int64(4), str(3)
memory usage: 61.2 KB


,Food_Item,Category,Calories (kcal),Protein (g),Carbohydrates (g),Fat (g),Fiber (g),Sugars (g),Sodium (mg),Cholesterol (mg),Meal_Type,Water_Intake (ml)
0,Scrambled Eggs (2 large),Protein/Dairy,180,12.0,2.0,14.0,0.0,1.0,180,370,Breakfast,250
1,Whole Wheat Toast (1 slice),Grain,80,4.0,14.0,1.0,2.0,2.0,140,0,Breakfast,0
2,Coffee (black),Beverage,5,0.3,0.0,0.1,0.0,0.0,5,0,Breakfast,0
3,Banana,Fruit,105,1.3,27.0,0.4,3.1,14.0,1,0,Breakfast,0
4,Grilled Chicken Salad,Meal/Protein,350,30.0,10.0,20.0,5.0,4.0,400,80,Lunch,500
5,Apple,Fruit,95,0.5,25.0,0.3,4.4,19.0,2,0,Snack,0
6,Salmon (4oz grilled),Protein/Fish,230,25.0,0.0,14.0,0.0,0.0,60,60,Dinner,250
7,Quinoa (1 cup cooked),Grain,222,8.0,39.0,3.6,5.0,1.0,13,0,Dinner,0
8,Steamed Broccoli (1 cup),Vegetable,55,3.7,11.2,0.6,5.1,2.0,40,0,Dinner,0
9,Greek Yogurt (plain 1 cup),Dairy,150,25.0,8.0,3.0,0.0,7.0,85,10,Snack,250


## 2. diet_recommendations_dataset.csv
> Profils patients + recommandations alimentaires

In [ ]:
# --- Diet Recommendation ---

diet_file = DATA_RAW / "diet_recommendations_dataset.csv"

if diet_file.exists():
# on_bad_lines='skip' permet de charger le fichier même s'il y a des erreurs
    df_diet = pd.read_csv(diet_file, on_bad_lines='skip', sep=',')
    
    print(f"✅ Chargé avec succès ({df_diet.shape[0]} lignes valides).")
    # Les types de données et valeurs nulles
    df_diet.info()
    print("\n Aperçu des données :")
    display(df_diet.head(5))
else:
    print(f" Fichier introuvable : {diet_file}")

✅ Chargé avec succès (1000 lignes valides).

📋 Colonnes et Types :


,Type,Exemple,Valeurs Nulles
Patient_ID,str,P0001,0
Age,int64,56,0
Gender,str,Male,0
Weight_kg,float64,58.4,0
Height_cm,int64,160,0
BMI,float64,22.8,0
Disease_Type,str,Obesity,204
Severity,str,Moderate,0
Physical_Activity_Level,str,Moderate,0
Daily_Caloric_Intake,int64,3079,0



 Aperçu des données :


,Patient_ID,Age,Gender,Weight_kg,Height_cm,BMI,Disease_Type,Severity,Physical_Activity_Level,Daily_Caloric_Intake,Cholesterol_mg/dL,Blood_Pressure_mmHg,Glucose_mg/dL,Dietary_Restrictions,Allergies,Preferred_Cuisine,Weekly_Exercise_Hours,Adherence_to_Diet_Plan,Dietary_Nutrient_Imbalance_Score,Diet_Recommendation
0,P0001,56,Male,58.4,160,22.8,Obesity,Moderate,Moderate,3079,173.3,133,116.3,NaN,Peanuts,Mexican,3.1,96.6,3.1,Balanced
1,P0002,69,Male,101.2,169,35.4,Diabetes,Mild,Moderate,3032,199.2,120,137.1,NaN,Peanuts,Chinese,4.5,63.2,0.6,Low_Carb
2,P0003,46,Female,63.5,173,21.2,Hypertension,Mild,Sedentary,1737,181.0,121,109.6,NaN,Peanuts,Chinese,3.8,57.5,4.6,Low_Sodium
3,P0004,32,Male,58.1,164,21.6,NaN,Mild,Moderate,2657,168.2,144,159.4,NaN,NaN,Mexican,4.3,54.5,0.4,Balanced
4,P0005,60,Male,79.5,197,20.5,Diabetes,Moderate,Sedentary,3496,200.4,172,182.3,Low_Sugar,NaN,Italian,9.8,78.2,4.7,Low_Carb


## 3. gym_members_exercise_tracking.csv
> Données biométriques + workout (Age, Gender, BPM, Calories, BMI...)

In [ ]:
df_gym = pd.read_csv(RAW / "gym_members_exercise_tracking.csv")
print(f"Shape : {df_gym.shape}")
display(df_gym.head(3))
print("\n--- Types ---")
print(df_gym.dtypes)
print("\n--- Valeurs nulles ---")
print(df_gym.isnull().sum())
print(f"\nDoublons : {df_gym.duplicated().sum()}")

## 4. gym_members_exercise_tracking_synthetic_data.csv
> Variante synthétique du dataset gym — mêmes colonnes

In [ ]:
df_gym_synth = pd.read_csv(RAW / "gym_members_exercise_tracking_synthetic_data.csv")
print(f"Shape : {df_gym_synth.shape}")
display(df_gym_synth.head(3))
print("\n--- Colonnes identiques au dataset gym ? ---")
print(set(df_gym.columns) == set(df_gym_synth.columns))
print(f"\nDoublons : {df_gym_synth.duplicated().sum()}")

## 5. 25.csv — Fitness Tracker
> Suivi quotidien : pas, humeur, calories, sommeil, poids

In [ ]:
df_tracker = pd.read_csv(RAW / "25.csv")
print(f"Shape : {df_tracker.shape}")
display(df_tracker.head(3))
print("\n--- Types ---")
print(df_tracker.dtypes)
print("\n--- Valeurs nulles ---")
print(df_tracker.isnull().sum())
print(f"\nDoublons : {df_tracker.duplicated().sum()}")
print(f"\nPériode : {df_tracker['date'].min()} -> {df_tracker['date'].max()}")

## 6. exercises.json
> Catalogue de 873 exercices (nom, niveau, muscles, équipement)

In [ ]:
with open(RAW / "exercises.json", encoding="utf-8") as f:
    exercises = json.load(f)
df_ex = pd.DataFrame(exercises)
print(f"Shape : {df_ex.shape}")
display(df_ex.head(3))
print("\n--- Types ---")
print(df_ex.dtypes)
print("\n--- Valeurs nulles ---")
print(df_ex.isnull().sum())
print(f"\nNiveaux : {df_ex['level'].value_counts().to_dict()}")
print(f"Categories : {df_ex['category'].value_counts().to_dict()}")

## Recap global

In [ ]:
recap = {
    "daily_food_nutrition": df_food.shape,
    "diet_recommendations": df_diet.shape,
    "gym_members": df_gym.shape,
    "gym_members_synthetic": df_gym_synth.shape,
    "fitness_tracker_25": df_tracker.shape,
    "exercises": df_ex.shape,
}
for name, shape in recap.items():
    print(f"{name:35s} -> {shape[0]:>6} lignes x {shape[1]:>2} colonnes")